# 1. Introduction

## Project Overview

This project implements a complete machine learning pipeline for multiclass vehicle classification using the Statlog Vehicle Silhouettes dataset from the UCI Machine Learning Repository.

The main objective is to compare different machine learning approaches and evaluate how preprocessing, dimensionality reduction, and ensemble methods affect model performance.

The project includes:

- Data preprocessing and feature scaling
- Correlation analysis and feature selection
- Logistic Regression with hyperparameter tuning
- Principal Component Analysis (PCA)
- Decision Tree classification
- Bagging ensembles
- Random Forest
- ROC curve analysis for multiclass classification
- Learning curve analysis

## Dataset Description

The Statlog Vehicle Silhouettes dataset contains geometric features extracted from vehicle silhouettes. The classification task consists of predicting the vehicle type based on shape-related numerical features.

Vehicle classes:

- bus
- opel
- saab
- van

The dataset contains 18 numerical features describing geometric properties such as compactness, circularity, rectangularity, skewness, kurtosis, and elongation.

# 2. Data Loading

## Data Import

The dataset is distributed across multiple `.dat` files. In this section, all files are loaded and combined into a single pandas DataFrame.

After loading the data, column names are assigned according to the dataset documentation.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
pd.set_option("display.max_columns", None)

plt.style.use("ggplot")

In [3]:
DATA_DIR = Path("/Users/olgah/code/python/vehicle-silhouettes-ml-pipeline/data")

data_files = sorted(DATA_DIR.glob("*.dat"))

data_files

[PosixPath('/Users/olgah/code/python/vehicle-silhouettes-ml-pipeline/data/xaa.dat'),
 PosixPath('/Users/olgah/code/python/vehicle-silhouettes-ml-pipeline/data/xab.dat'),
 PosixPath('/Users/olgah/code/python/vehicle-silhouettes-ml-pipeline/data/xac.dat'),
 PosixPath('/Users/olgah/code/python/vehicle-silhouettes-ml-pipeline/data/xad.dat'),
 PosixPath('/Users/olgah/code/python/vehicle-silhouettes-ml-pipeline/data/xae.dat'),
 PosixPath('/Users/olgah/code/python/vehicle-silhouettes-ml-pipeline/data/xaf.dat'),
 PosixPath('/Users/olgah/code/python/vehicle-silhouettes-ml-pipeline/data/xag.dat'),
 PosixPath('/Users/olgah/code/python/vehicle-silhouettes-ml-pipeline/data/xah.dat'),
 PosixPath('/Users/olgah/code/python/vehicle-silhouettes-ml-pipeline/data/xai.dat')]

In [4]:
dfs = []

for file in data_files:
    df_part = pd.read_csv(
        file,
        sep=r"\s+",
        header=None
    )
    
    dfs.append(df_part)

df = pd.concat(dfs, ignore_index=True)

df.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18
0,95,48,83,178,72,10,162,42,20,159,176,379,184,70,6,16,187,197,van
1,91,41,84,141,57,9,149,45,19,143,170,330,158,72,9,14,189,199,van
2,104,50,106,209,66,10,207,32,23,158,223,635,220,73,14,9,188,196,saab
3,93,41,82,159,63,9,144,46,19,143,160,309,127,63,6,10,199,207,van
4,85,44,70,205,103,52,149,45,19,144,241,325,188,127,9,11,180,183,bus


In [5]:
columns = [
    "compactness",
    "circularity",
    "distance_circularity",
    "radius_ratio",
    "pr_axis_aspect_ratio",
    "max_length_aspect_ratio",
    "scatter_ratio",
    "elongatedness",
    "pr_axis_rectangularity",
    "max_length_rectangularity",
    "scaled_variance_major",
    "scaled_variance_minor",
    "scaled_radius_gyration",
    "skewness_major",
    "skewness_minor",
    "kurtosis_major",
    "kurtosis_minor",
    "hollows_ratio",
    "class"
]

df.columns = columns

df.head()

,compactness,circularity,distance_circularity,radius_ratio,pr_axis_aspect_ratio,max_length_aspect_ratio,scatter_ratio,elongatedness,pr_axis_rectangularity,max_length_rectangularity,scaled_variance_major,scaled_variance_minor,scaled_radius_gyration,skewness_major,skewness_minor,kurtosis_major,kurtosis_minor,hollows_ratio,class
0,95,48,83,178,72,10,162,42,20,159,176,379,184,70,6,16,187,197,van
1,91,41,84,141,57,9,149,45,19,143,170,330,158,72,9,14,189,199,van
2,104,50,106,209,66,10,207,32,23,158,223,635,220,73,14,9,188,196,saab
3,93,41,82,159,63,9,144,46,19,143,160,309,127,63,6,10,199,207,van
4,85,44,70,205,103,52,149,45,19,144,241,325,188,127,9,11,180,183,bus


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 846 entries, 0 to 845
Data columns (total 19 columns):
 #   Column                     Non-Null Count  Dtype
---  ------                     --------------  -----
 0   compactness                846 non-null    int64
 1   circularity                846 non-null    int64
 2   distance_circularity       846 non-null    int64
 3   radius_ratio               846 non-null    int64
 4   pr_axis_aspect_ratio       846 non-null    int64
 5   max_length_aspect_ratio    846 non-null    int64
 6   scatter_ratio              846 non-null    int64
 7   elongatedness              846 non-null    int64
 8   pr_axis_rectangularity     846 non-null    int64
 9   max_length_rectangularity  846 non-null    int64
 10  scaled_variance_major      846 non-null    int64
 11  scaled_variance_minor      846 non-null    int64
 12  scaled_radius_gyration     846 non-null    int64
 13  skewness_major             846 non-null    int64
 14  skewness_minor             846 non-nu

In [7]:
df.describe()

,compactness,circularity,distance_circularity,radius_ratio,pr_axis_aspect_ratio,max_length_aspect_ratio,scatter_ratio,elongatedness,pr_axis_rectangularity,max_length_rectangularity,scaled_variance_major,scaled_variance_minor,scaled_radius_gyration,skewness_major,skewness_minor,kurtosis_major,kurtosis_minor,hollows_ratio
count,846.000000,846.000000,846.000000,846.000000,846.000000,846.000000,846.000000,846.000000,846.000000,846.000000,846.000000,846.000000,846.00000,846.000000,846.000000,846.000000,846.000000,846.000000
mean,93.678487,44.861702,82.088652,168.940898,61.693853,8.567376,168.839243,40.933806,20.582742,147.998818,188.625296,439.911348,174.70331,72.462175,6.377069,12.599291,188.932624,195.632388
std,8.234474,6.169866,15.771533,33.472183,7.888251,4.601217,33.244978,7.811560,2.592138,14.515652,31.394837,176.692614,32.54649,7.486974,4.918353,8.931240,6.163949,7.438797
min,73.000000,33.000000,40.000000,104.000000,47.000000,2.000000,112.000000,26.000000,17.000000,118.000000,130.000000,184.000000,109.00000,59.000000,0.000000,0.000000,176.000000,181.000000
25%,87.000000,40.000000,70.000000,141.000000,57.000000,7.000000,146.250000,33.000000,19.000000,137.000000,167.000000,318.250000,149.00000,67.000000,2.000000,5.000000,184.000000,190.250000
50%,93.000000,44.000000,80.000000,167.000000,61.000000,8.000000,157.000000,43.000000,20.000000,146.000000,178.500000,364.000000,173.00000,71.500000,6.000000,11.000000,188.000000,197.000000
75%,100.000000,49.000000,98.000000,195.000000,65.000000,10.000000,198.000000,46.000000,23.000000,159.000000,217.000000,587.000000,198.00000,75.000000,9.000000,19.000000,193.000000,201.000000
max,119.000000,59.000000,112.000000,333.000000,138.000000,55.000000,265.000000,61.000000,29.000000,188.000000,320.000000,1018.000000,268.00000,135.000000,22.000000,41.000000,206.000000,211.000000


In [8]:
df["class"].value_counts()

class
bus     218
saab    217
opel    212
van     199
Name: count, dtype: int64

## Data Loading Summary

The dataset was successfully loaded from multiple `.dat` files and combined into a single DataFrame.

The final dataset contains 846 observations, 18 numerical features, and one target variable.

There are no missing values in the dataset. The target classes are relatively balanced, which allows us to use accuracy and macro F1-score for model evaluation.

# 3. Exploratory Data Analysis

## Exploratory Data Analysis

In this section, the dataset structure and feature distributions are explored.

The following analyses are performed:

- Dataset shape inspection
- Feature type analysis
- Missing value detection
- Class distribution analysis
- Correlation analysis
- Feature distribution visualization

The purpose of EDA is to better understand the dataset before model training and preprocessing.

# 4. Data Preprocessing

## Data Preprocessing

Before training machine learning models, the dataset is preprocessed.

The preprocessing pipeline includes:

- Train-test split
- Feature scaling using `StandardScaler`
- Correlation analysis
- Removal of highly correlated features

Feature scaling is necessary because Logistic Regression and PCA are sensitive to feature magnitude.

Highly correlated features are removed to reduce multicollinearity and improve model stability.

# 5. Logistic Regression

## Logistic Regression Baseline

Logistic Regression is used as the baseline classification model.

Hyperparameter tuning is performed using `GridSearchCV` with cross-validation to identify the optimal regularization parameters.

Model performance is evaluated using:

- Accuracy
- F1-score
- Multiclass ROC curves

The results obtained in this section serve as a baseline for comparison with more advanced models.

# 6. PCA

## Principal Component Analysis (PCA)

Principal Component Analysis is applied to reduce feature dimensionality and analyze variance distribution across components.

Before applying PCA, the data is scaled using `StandardScaler`.

The following steps are performed:

- PCA fitting on training data only
- Explained variance analysis
- Selection of the optimal number of components
- Logistic Regression training on PCA-transformed data

The purpose of PCA is to investigate whether dimensionality reduction improves model performance and reduces feature redundancy.

# 7. Decision Tree

## Decision Tree Classification

A Decision Tree classifier is trained and optimized using cross-validation.

The optimal tree depth is selected using `GridSearchCV`.

Model performance is evaluated using:

- Accuracy
- F1-score

The Decision Tree model is compared with Logistic Regression to analyze the impact of nonlinear decision boundaries.

# 8. Bagging

## Bagging Ensembles

Bagging ensembles are applied to improve model stability and reduce variance.

Two ensemble configurations are investigated:

- Bagging with Logistic Regression
- Bagging with Decision Trees

The influence of the number of estimators on model performance is analyzed using:

- Accuracy curves
- F1-score curves

The goal is to determine whether ensemble methods improve classification performance compared to single models.

# 9. Random Forest

## Random Forest Classification

Random Forest is trained as an advanced ensemble learning method based on multiple decision trees.

The influence of the number of trees on model quality is analyzed.

Model evaluation includes:

- Accuracy
- F1-score

Performance trends are visualized to identify the optimal number of trees and evaluate ensemble effectiveness.

# 10. Learning Curves

## Learning Curve Analysis

Learning curves are used to analyze how model performance changes as the size of the training dataset increases.

The training data is divided into multiple subsets of increasing size.

For each subset, models are trained and evaluated using:

- Accuracy
- F1-score

This analysis helps determine:

- Whether the models benefit from additional data
- Whether underfitting or overfitting is present
- How model stability changes with dataset size

# 11. Final Conclusions

## Final Conclusions

In this section, the performance of all trained models is summarized and compared.

The analysis focuses on:

- The impact of preprocessing
- The effectiveness of PCA
- The benefits of ensemble learning
- Model stability and generalization performance

The best-performing model is identified based on evaluation metrics and overall behavior across experiments.